# PRÉ-PROJETO DE PESQUISA OPERACIONAL — python-mip com HiGHS, CBC ou Gurobi

## Otimização Operacional do Agendamento de Bombas e Distribuição de Água Tratada via Programação Linear Inteira Mista (MILP)

- Escolha o solver alterando apenas a variável `SOLVER` na célula de configuração.
- A célula inicial instala python-mip, HiGHS, CBC e a interface Python do Gurobi.
- Para usar Gurobi pelo `python-mip`, é necessária uma instalação nativa do Gurobi e uma licença compatível com a API C; a licença restrita do pacote pip `gurobipy` funciona pela interface `gurobipy` (como Pyomo), mas não pelo backend Gurobi do `python-mip`.
- Defina `GUROBI_HOME` para a pasta da instalação nativa antes de iniciar o kernel e reinicie-o. No macOS/Linux, essa pasta deve conter `lib/libgurobi*.dylib` ou `lib/libgurobi*.so`; no Windows, `bin/gurobi*.dll`.

In [15]:
%pip install --upgrade "mip==2.0.0" "highspy>=1.8" cbcbox gurobipy

Note: you may need to restart the kernel to use updated packages.


In [16]:
import mip
from mip.exceptions import InterfacingError
from mip import Model, xsum, BINARY, CONTINUOUS, MINIMIZE, OptimizationStatus
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Conjuntos de Índices

T = 24
horas = range(T)
bombas = range(1, 7)
reservatorios = range(1, 6)
setores = range(1, 13)

# 2. Topologia do Sistema (Arcos permitidos conforme o diagrama)
# Arcos (Bomba -> Reservatório)
arcos_PR = [
    (1,1), 
    (2,1), (2,2), (2,3), 
    (3,2), 
    (4,3), (4,4), 
    (5,5), 
    (6,4), (6,5)
]

# Arcos (Reservatório -> Setor de Consumo)
arcos_RD = [
    (1,1), (1,2), (1,3),
    (2,2), (2,4), (2,5),
    (3,4), (3,6), (3,7),
    (4,4), (4,8), (4,9), (4,10), (4,12),
    (5,7), (5,10), (5,11), (5,12)
]

# 3. Parâmetros Econômicos (Tarifa Horossazonal - R$/kWh)
# Tarifa fora de ponta: R$ 0.4525 | Tarifa de ponta (18h00 às 20h59 ): R$ 1.4048
C = {t: 1.4048 if 18 <= t < 21 else 0.4525 for t in horas}

# 4. Parâmetros das bombas: (potência de referência em kW, vazão mínima e máxima em m³/h)
# Potências estimadas no ponto de vazão máxima, com 30 m de altura manométrica e 80% de eficiência global.
# O objetivo considera essa potência constante durante cada período ligado.
param_bombas = {
    1: (240, 1170, 2340),
    2: (450, 2180, 4370),
    3: (240, 1170, 2340),
    4: (290, 1400, 2810),
    5: (190,  940, 1870),
    6: (240, 1170, 2340)
}
Q_ETA_max = 9400  # m³/h; limite de vazão agregada da ETA

# 5. Parâmetros dos Reservatórios: (volume mínimo, máximo e inicial), em m³
# O volume mínimo é a reserva de segurança; o volume inicial define o estoque no início do horizonte.
param_res = {
    1: (2100, 14000, 4200),
    2: (1400,  9400, 2800),
    3: (1400,  9400, 2800),
    4: (1400,  9400, 2800),
    5: ( 700,  4700, 1400)
}

# 6. Demandas-base dos setores (m³/h); total: 5195 m³/h
base_demanda = {
    1: 312,  2: 468,  3: 390,  4: 624,
    5: 390,  6: 312,  7: 546,  8: 468,
    9: 312, 10: 702, 11: 390, 12: 281
}

# Curva horária de demanda de verão; cada fator multiplica a demanda-base dos setores.
# Demanda média diária: 8312 m³/h; pico: 17663 m³/h às 18h.
curva_verao = [
    0.8, 0.5, 0.3, 0.4, 0.6, 1.0,   # 00h às 05h: Queda brusca do consumo na madrugada
    1.8, 2.0, 1.9, 1.7, 1.5, 1.5,   # 06h às 11h: Pico pesado da manhã
    1.4, 1.5, 1.6, 1.7, 1.8, 2.9,   # 12h às 17h: Consumo alto e contínuo
    3.4, 3.0, 2.6, 2.0, 1.5, 1.0    # 18h às 23h: Pico máximo (chegada da praia + banhos)
]

D = {}
for d in setores:
    for t in horas:
        D[d, t] = round(base_demanda[d] * curva_verao[t], 2)



In [17]:
# Altere somente esta variável: "highs", "cbc" ou "gurobi".
SOLVER = "cbc"
SOLVER_BACKENDS = {
    "highs": mip.HIGHS,
    "cbc": mip.CBC,
    "gurobi": mip.GRB,
}
TEMPO_LIMITE = 300
GAP_RELATIVO = 0.003
THREADS = 0

if SOLVER not in SOLVER_BACKENDS:
    raise ValueError(
        f"SOLVER inválido: {SOLVER!r}. Escolha entre {', '.join(SOLVER_BACKENDS)}."
    )

if SOLVER == "gurobi":
    import os
    import sys
    from pathlib import Path

    gurobi_home = os.environ.get("GUROBI_HOME")
    if not gurobi_home:
        raise RuntimeError(
            "O backend Gurobi do python-mip usa a API C. Configure GUROBI_HOME "
            "para uma instalação nativa do Gurobi com licença compatível e reinicie o kernel. "
            "A licença restrita instalada pelo pacote pip gurobipy só pode ser usada pela interface gurobipy."
        )

    gurobi_home = Path(gurobi_home).expanduser()
    diretorio_bibliotecas = gurobi_home / ("bin" if os.name == "nt" else "lib")
    padrao_biblioteca = "gurobi*.dll" if os.name == "nt" else "libgurobi*"
    bibliotecas_nativas = {p.resolve() for p in diretorio_bibliotecas.glob(padrao_biblioteca)}
    if not bibliotecas_nativas:
        raise RuntimeError(
            f"Nenhuma biblioteca nativa do Gurobi ({padrao_biblioteca}) encontrada em "
            f"{diretorio_bibliotecas}. Confira o valor de GUROBI_HOME."
        )

    modulo_gurobi_mip = sys.modules.get("mip.gurobi")
    if modulo_gurobi_mip is not None:
        biblioteca_carregada = getattr(modulo_gurobi_mip, "lib_path", None)
        if biblioteca_carregada and Path(biblioteca_carregada).resolve() not in bibliotecas_nativas:
            raise RuntimeError(
                "O python-mip já carregou outra biblioteca Gurobi neste kernel. "
                "Reinicie o kernel depois de configurar GUROBI_HOME."
            )


In [18]:
# Constrói o mesmo MILP para a otimização tarifária ou para o cenário-base.
def construir_modelo(custos, nome, vazao_ponta=None):
    try:
        m = Model(
            name=nome,
            sense=MINIMIZE,
            solver_name=SOLVER_BACKENDS[SOLVER],
        )
    except InterfacingError as exc:
        if SOLVER == "gurobi":
            raise RuntimeError(
                "O python-mip não conseguiu iniciar o Gurobi pela API C. Verifique se GUROBI_HOME "
                "aponta para a instalação nativa correta e se a licença é compatível com a API C."
            ) from exc
        raise
    m.max_seconds = TEMPO_LIMITE
    m.max_mip_gap = GAP_RELATIVO
    m.threads = THREADS
    m.verbose = 0

    x = {
        (p, t): m.add_var(var_type=BINARY, name=f"x_{p}_{t}")
        for p in bombas for t in horas
    }
    q = {
        (p, t): m.add_var(var_type=CONTINUOUS, lb=0, name=f"q_{p}_{t}")
        for p in bombas for t in horas
    }
    f_pr = {
        (p, r, t): m.add_var(var_type=CONTINUOUS, lb=0, name=f"f_pr_{p}_{r}_{t}")
        for p, r in arcos_PR for t in horas
    }
    f_rd = {
        (r, d, t): m.add_var(var_type=CONTINUOUS, lb=0, name=f"f_rd_{r}_{d}_{t}")
        for r, d in arcos_RD for t in horas
    }
    V = {
        (r, t): m.add_var(var_type=CONTINUOUS, lb=0, name=f"V_{r}_{t}")
        for r in reservatorios for t in horas
    }

    m.objective = xsum(
        custos[t] * param_bombas[p][0] * x[p, t]
        for p in bombas for t in horas
    )

    for p in bombas:
        Q_min, Q_max = param_bombas[p][1], param_bombas[p][2]
        for t in horas:
            m += q[p, t] >= Q_min * x[p, t], f"Bomba_Min_{p}_{t}"
            m += q[p, t] <= Q_max * x[p, t], f"Bomba_Max_{p}_{t}"
            m += q[p, t] == xsum(
                f_pr[p, r, t] for pp, r in arcos_PR if pp == p
            ), f"Fluxo_Bomba_{p}_{t}"

    for t in horas:
        m += xsum(q[p, t] for p in bombas) <= Q_ETA_max, f"Capacidade_ETA_{t}"
        if vazao_ponta is not None and t in vazao_ponta:
            m += xsum(q[p, t] for p in bombas) == vazao_ponta[t], f"Vazao_Ponta_{t}"

    for d in setores:
        for t in horas:
            m += xsum(
                f_rd[r, setor, t]
                for r, setor in arcos_RD if setor == d
            ) == D[d, t], f"Demanda_{d}_{t}"

    for r in reservatorios:
        V_min, V_max, V_inicial = param_res[r]
        for t in horas:
            vazao_in = xsum(
                f_pr[p, reservatorio, t]
                for p, reservatorio in arcos_PR if reservatorio == r
            )
            vazao_out = xsum(
                f_rd[reservatorio, d, t]
                for reservatorio, d in arcos_RD if reservatorio == r
            )
            volume_anterior = V_inicial if t == 0 else V[r, t - 1]
            m += V[r, t] == volume_anterior + vazao_in - vazao_out, f"Balanco_{r}_{t}"
            m += V[r, t] >= V_min, f"Vmin_{r}_{t}"
            m += V[r, t] <= V_max, f"Vmax_{r}_{t}"
        m += V[r, 23] >= V_inicial, f"Recuperacao_{r}"

    return m, x, q, f_pr, f_rd, V

model, x, q, f_pr, f_rd, V = construir_modelo(
    custos=C,
    nome=f"Agendamento_Bombas_{SOLVER.upper()}",
)


## Cenário-base com tarifa fixa

Este cenário mantém as mesmas demandas, capacidades e balanços dos reservatórios, mas usa uma tarifa uniforme na função objetivo. Nas horas de ponta, fixa o bombeamento total na demanda agregada da hora, limitado à capacidade máxima da ETA. Depois, o custo da programação-base é recalculado com as tarifas horárias reais. A otimização tarifária original é executada em seguida, sem alteração.

In [ ]:
# Na ponta, bombeia a demanda agregada; se ela exceder a capacidade da ETA,
# usa a capacidade máxima disponível naquele horário.
horas_ponta = [t for t in horas if C[t] == max(C.values())]
vazao_base_ponta = {
    t: min(sum(D[d, t] for d in setores), Q_ETA_max)
    for t in horas_ponta
}
modelo_base, x_base, q_base, f_pr_base, f_rd_base, V_base = construir_modelo(
    custos={t: 1.0 for t in horas},
    nome=f"Cenario_Base_{SOLVER.upper()}",
    vazao_ponta=vazao_base_ponta,
)
status_base = modelo_base.optimize(max_seconds=TEMPO_LIMITE)
base_solucao_disponivel = status_base in (
    OptimizationStatus.OPTIMAL,
    OptimizationStatus.FEASIBLE,
)
if not base_solucao_disponivel:
    raise RuntimeError(
        f"O cenário-base não encontrou solução viável. Status: {status_base}"
    )

consumo_base_kwh = sum(
    param_bombas[p][0] * x_base[p, t].x
    for p in bombas for t in horas
)

# Avalia a programação-base com a tarifa horária real, sem reotimizá-la.
custo_base_real = sum(
    C[t] * param_bombas[p][0] * x_base[p, t].x
    for p in bombas for t in horas
)
print("[CENÁRIO-BASE: TARIFA FIXA]")
print("Status:", status_base)
print("Vazão-alvo na ponta (m³/h):", vazao_base_ponta)
print(f"Consumo de energia: {consumo_base_kwh:.2f} kWh")
print(f"Custo dessa programação com as tarifas reais: R$ {custo_base_real:.2f}")

In [19]:
# Otimização com o solver selecionado na variável SOLVER
import time
model.verbose = 1
tempo_inicio = time.perf_counter()
status_terminal = model.optimize(max_seconds=TEMPO_LIMITE)
tempo_execucao = time.perf_counter() - tempo_inicio
solucao_disponivel = status_terminal in (
    OptimizationStatus.OPTIMAL,
    OptimizationStatus.FEASIBLE,
)

incumbente = model.objective_value if solucao_disponivel else None
limite_inferior = model.objective_bound if solucao_disponivel else None
gap_final = model.gap if solucao_disponivel else None
if gap_final is not None and not np.isfinite(gap_final):
    gap_final = None

print(f"\n[RESULTADOS DO {SOLVER.upper()} VIA PYTHON-MIP]")
print("Status:", status_terminal)
print(f"Custo/solução incumbente: R$ {incumbente:.2f}" if incumbente is not None else "Incumbente: não disponível")
print(f"Melhor limite inferior: R$ {limite_inferior:.2f}" if limite_inferior is not None else "Limite inferior: não disponível")
print(f"Gap relativo final: {gap_final:.2%}" if gap_final is not None else "Gap relativo final: não disponível")
print(f"Tempo: {tempo_execucao:.2f} s")

if solucao_disponivel:
    print(f"\n[SUCESSO] Solução encontrada pelo {SOLVER.upper()} via python-mip!")
else:
    print(f"\nModelo inviável ou sem solução viável encontrada pelo {SOLVER.upper()}.")



▶ Problem loading

  Problem: Agendamento_Bombas_CBC
  |Rows| = 1109   |Cols| = 1080   |NZ| = 2688
  Variables: 144 binary, 0 integer, 936 continuous

  Coefficient ranges:
    Matrix  ∈ [     1,   4370]   κ ≈   4370
    Cost    ∈ [85.975, 632.16]   κ ≈ 7.3528
    Bounds  ∈ [     1,      1]   κ ≈      1
    RHS     ∈ [  84.3,  14000]   κ ≈ 166.07

✔ Problem loaded — 1109 rows, 1080 cols, 2688 NZ, 144 integer

▶ Non-default parameters

  seconds  1e+08 → 300
  allowableGap  1e-06 → 1e-10
  ratioGap  0.0001 → 0.003
  lpTimeFreq  5 → 0

▶ Root LP relaxation

  Bound tightening: 942 bounds tightened (168 fixed) in 0.001 s.
  Clique strengthening: no changes (0.00s)
  Coefficient strengthening: no changes (0.00s)
  Row reductions: 168 rows removed (168 all-fixed, 0 duplicate, 0 parallel), 0 row bounds tightened (0.00s)
  LP auto: recommended primal_sprint

▶ Preprocessing


  Processed model: 526 rows, 712 cols, 1712 NZ
                   144 integer (144 binary)

  Conflict graph: built i

In [ ]:
# Compara as duas programações usando o mesmo preço horário real.
if solucao_disponivel:
    consumo_otimizado_kwh = sum(
        param_bombas[p][0] * x[p, t].x
        for p in bombas for t in horas
    )
    custo_otimizado_real = sum(
        C[t] * param_bombas[p][0] * x[p, t].x
        for p in bombas for t in horas
    )
    comparacao_custos = pd.DataFrame([
        {
            "Cenário": "Base (tarifa fixa)",
            "Energia (kWh)": consumo_base_kwh,
            "Custo com tarifa real (R$)": custo_base_real,
        },
        {
            "Cenário": "Otimização (tarifa horária)",
            "Energia (kWh)": consumo_otimizado_kwh,
            "Custo com tarifa real (R$)": custo_otimizado_real,
        },
    ])
    display(comparacao_custos)
    economia = custo_base_real - custo_otimizado_real
    percentual_economia = 100 * economia / custo_base_real if custo_base_real else 0
    print(f"Diferença de custo: R$ {economia:.2f} ({percentual_economia:.2f}%)")
else:
    print("A comparação tarifária depende de uma solução viável da otimização.")

# GRÁFICOS E RESULTADOS

In [ ]:
# Célula 3: Extração de Resultados e Visualização (Matplotlib) - Eixo 0 a 24
if solucao_disponivel:
    import matplotlib.pyplot as plt

    # 1. Preparação dos dados
    # Eixo X da linha do tempo contínua com 25 marcos (0 a 24)
    horas_plot = list(range(25))
    
    # Vazão total bombeada (24 blocos de 1 hora)
    vazao_total = [sum(q[p, t].x for p in bombas) for t in horas]
    
    # Preço da energia (Adicionamos o último valor repetido apenas para o gráfico desenhar o degrau até o final da hora 24)
    preco_energia = [C[t] for t in horas]
    preco_energia.append(preco_energia[-1]) 

    # 2. Configuração da Figura
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

    # --- GRÁFICO 1: Operação das Bombas vs Tarifa ---
    ax1.bar(horas, vazao_total, width=1, align='edge', color='#2196F3', alpha=0.8, edgecolor='white', label='Volume Bombeado (m³/h)')
    ax1.set_ylabel('Vazão Total (m³/h)', color='#0b5394', fontweight='bold')
    ax1.tick_params(axis='y', labelcolor='#0b5394')
    ax1.set_title('Perfil de Acionamento das Bombas e Custo da Energia', fontsize=14)
    ax1_twin = ax1.twinx()
    ax1_twin.step(horas_plot, preco_energia, where='post', color='#f44336', linewidth=2, label='Tarifa de Energia (R$/kWh)')
    ax1_twin.set_ylabel('Tarifa (R$/kWh)', color='#990000', fontweight='bold')
    ax1_twin.tick_params(axis='y', labelcolor='#990000')
    ax1.axvspan(17, 20, color='red', alpha=0.1) 
    
    # --- GRÁFICO 2: Comportamento dos Reservatórios (m³ Absoluto) ---
    cores_res = ['#4CAF50', '#8BC34A', '#CDDC39', '#FFEB3B', '#FFC107']
    for r in reservatorios:
        V_min, V_max, V_inicial = param_res[r]
      
        vol_absoluto = [V_inicial] + [V[r, t].x for t in horas]
        
        ax2.plot(horas_plot, vol_absoluto, marker='o', markersize=4, linewidth=2, color=cores_res[r-1], label=f'Reservatório {r}')
    
    # Sombreamento do horário de ponta
    ax2.axvspan(17, 20, color='red', alpha=0.1)
    
    ax2.set_xlabel('Hora do Dia (0 = Início do Dia, 24 = Final do Dia)', fontweight='bold')
    ax2.set_ylabel('Volume Armazenado (m³)', fontweight='bold')
    ax2.set_title('Dinâmica de Carga e Descarga das "Baterias Hídricas" (Em Valores Absolutos)', fontsize=14)
    ax2.set_xticks(range(0, 25, 2))
    ax2.set_xlim(0, 24)
    
    ax2.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
    ax1_twin.legend(loc='upper right', bbox_to_anchor=(1.25, 1.15))
    ax2.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

# 3. Gráfico do Nível Geral de cada Reservatório (Em Porcentagem - 0h a 24h)
   
    horas_plot = list(range(25)) 
    
    plt.figure(figsize=(12, 6))
    cores = ['#4CAF50', '#8BC34A', '#CDDC39', '#FFEB3B', '#FFC107']
    
    for r in reservatorios:
        V_min, V_max, V_inicial = param_res[r]
        vol_absoluto = [V_inicial] + [V[r, t].x for t in horas]
        nivel_percent = [(v / V_max) * 100 for v in vol_absoluto]
        plt.plot(horas_plot, nivel_percent, marker='o', linewidth=2, color=cores[r-1], label=f'Reservatório {r}')

    # Destaque do horário de ponta: Começa na hora 17 (17:00) e termina na hora 20 (20:00)
    plt.axvspan(17, 20, color='red', alpha=0.1, label='Horário de Ponta (17h-20h)')
    
    # Linha pontilhada de referência no limite de 100% da capacidade
    plt.axhline(100, color='black', linestyle=':', alpha=0.5)
    
    plt.title('Evolução do Nível de Armazenamento por Reservatório (%)', fontsize=14)
    plt.xlabel('Hora do Dia (0 = Início do Dia, 24 = Final do Dia)', fontweight='bold')
    plt.ylabel('Volume Armazenado (%)', fontweight='bold')
    plt.xticks(range(0, 25, 2)) 
    plt.xlim(0, 24)
    plt.ylim(0, 105) 
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

else:
    print("Execute a otimização com sucesso antes de gerar os gráficos.")
